In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch import Tensor
from torchvision import transforms
import h5py

import matplotlib.pyplot as plt
import matplotlib.animation as animation

from IPython.display import HTML

import gc
import math
import random

# for deleting old best models
import os
import shutil
import glob

from dotenv import load_dotenv


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device is {device}")

We want to lazy-load the dataset, in an on-demand fashion, as the dataset is too large to be loaded into RAM all at once.
Each data item requires:
- External preprocessing, to conduct event-framing, resize DVS and APS frames to 224 * 224 and to clip only the relevant sections of the dataset.
    - This will be done by the script provided by ddd20-utils. DONE, see prepare_data.sh
- Internal preprocessing:
    - To normalise DVS frames,
    - To normalise APS frames,
    - (Optionally) to prune some of the frames/batches where the steering angle is low,
    - (Optionally) to prune the frames/batches where the vehicle speed is low.

The flow will be as such:
- The external script will conduct external preprocessing, and put all the .hdf5 files in one directory. DONE
- We must glob that directory, iterate over all all the files, DONE
- (Optionally) We must conduct the pruning for low steering angle/low speed before putting them into the dataset.
    - This requires the re-saving of the files in an external directory for lazy loading to be possible.
- We must then normalise the DVS and the APS. DONE
- We must create a system where the __iter__ method can nicely return something along all datasets. DONE



In [ ]:
load_dotenv("config.env")

# --- Dataset parameters ---
USE_DVS = bool(os.getenv("EXPORT_DVS"))
USE_APS = bool(os.getenv("EXPORT_APS"))
BINSIZE_MS = int(os.getenv("BINSIZE_MS"))
HEIGHT = int(os.getenv("HEIGHT"))
WIDTH = int(os.getenv("WIDTH"))
PRECISION = int(os.getenv("PRECISION"))
DTYPE = np.float16 if PRECISION == 16 else np.float32

APS_FRAME_NAME = f"aps_frame_{HEIGHT}x{WIDTH}"
DVS_FRAME_NAME = f"dvs_frame_{HEIGHT}x{WIDTH}"


# --- Location parameters ---
DATASET_DIR = os.getenv("OUT_DIR")

# --- Preprocessing parameters ---
PERFORM_PRUNING = bool(os.getenv("PERFORM_PRUNING"))
# km/h for speed to be considered high enough
SPEED_THRESHOLD = float(os.getenv("SPEED_THRESHOLD"))
# absolute value for degree of steering to be considered high enough
STEERING_THRESHOLD = float(os.getenv("STEERING_THRESHOLD"))
STEERING_DISCARD_CHANCE = float(os.getenv("STEERING_DISCARD_CHANCE"))

# --- Training parameters ---
TRAINING_SPLIT = float(os.getenv("TRAINING_SPLIT"))
EPOCHS = int(os.getenv("EPOCHS"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE"))
LEARNING_RATE = float(os.getenv("LEARNING_RATE"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY"))


In [4]:
# Randomly allocate a percentage of the data to the training set
TRAINING_DIR = f"{DATASET_DIR}/Training"
TEST_DIR = f"{DATASET_DIR}/Test"

# Try to create directory, shuffle the files back to the main directory if it already exists
try:
    os.mkdir(TRAINING_DIR)
except FileExistsError:
    for files in glob.glob(TRAINING_DIR + "/*.hdf5"):
        os.rename(files, f"{DATASET_DIR}/{os.path.basename(files)}")
    # delete everything that remains, a.k.a. pruned files
    for file in glob.glob(TRAINING_DIR + "/*"):
        if os.path.isdir(file):
            shutil.rmtree(file)
        else:
            os.remove(file)

try:
    os.mkdir(TEST_DIR)
except FileExistsError:
    for files in glob.glob(TEST_DIR + "/*"):
         os.rename(files, f"{DATASET_DIR}/{os.path.basename(files)}")


In [5]:
data_files = glob.glob(f"{DATASET_DIR}/*.hdf5")

# Randomly shuffle the data files
random.shuffle(data_files)

split_idx = int(len(data_files) * TRAINING_SPLIT)
train_files = data_files[:split_idx]
test_files = data_files[split_idx:]

for file in train_files:
    os.rename(file, f"{TRAINING_DIR}/{os.path.basename(file)}")

for file in test_files:
    os.rename(file, f"{TEST_DIR}/{os.path.basename(file)}")


In [ ]:
# Only prune training data
if PERFORM_PRUNING:
    PRUNE_SPEED = bool(os.getenv("SPEED_PRUNING"))
    PRUNE_STEERING = bool(os.getenv("STEERING_PRUNING"))
    
    PRUNED_DATASET_PATH = TRAINING_DIR + "/Pruned-Recordings"
    os.mkdir(PRUNED_DATASET_PATH)
    
    training_files = glob.glob(f"{TRAINING_DIR}/*.hdf5")
    for file in training_files:
        print(f"Pruning file: {file}")
        discarded_batches = []
        with h5py.File(file, "r") as f:
            speeds = np.array(f["vehicle_speed"])
            steering_angles = np.array(f["steering_wheel_angle"])
            file_len = len(speeds)
            for i in range(0, len(speeds), BATCH_SIZE):
                if PRUNE_SPEED:
                    if np.average(speeds[i:i+BATCH_SIZE]) < SPEED_THRESHOLD:
                        discarded_batches.append(i)
                        continue
                if PRUNE_STEERING:
                    if np.average(np.absolute(steering_angles[i:i+BATCH_SIZE])) < STEERING_THRESHOLD and random.random() < STEERING_DISCARD_CHANCE:
                        discarded_batches.append(i)

            indices_to_remove = []
            for idx in discarded_batches:
                indices_to_remove.extend(range(idx, idx + BATCH_SIZE))
            indices_to_keep = np.setdiff1d(np.arange(file_len), indices_to_remove)
            renamed_file = f"{PRUNED_DATASET_PATH}/{os.path.splitext(os.path.basename(file))[0]}_pruned.hdf5"
            with h5py.File(renamed_file, "w") as new_f:
                for dset_name in f.keys():
                    if len(f[dset_name]) != file_len:
                        # print(f"Dataset {dset_name} has a different length than the rest of the data, skipping.")
                        # in practice this is just test_idxs, and train_idxs, which are not needed
                        continue
                    full_data = np.array(f[dset_name])
                    new_f.create_dataset(dset_name, data=full_data[indices_to_keep])
            
            print(f"Discarded {len(discarded_batches)} batches, as their average was below the thresholds, totalling {len(indices_to_remove)} frames.")
            print(f"{len(indices_to_keep)} frames remain, with average speed {np.average(speeds[indices_to_keep])} and average absolute steering {np.average(np.absolute(steering_angles[indices_to_keep]))}") 

    TRAINING_DIR = PRUNED_DATASET_PATH

In [7]:
# adapted from https://stackoverflow.com/questions/74375033/pytorch-dataloader-for-multiple-files-with-sliding-window
class LazyDDD20Dataset(Dataset):
    def __init__(self, data_folder, *, transform=None, target_transform=None, window_size=1, use_aps=True, use_dvs=True):
        if not (use_aps or use_dvs):
            raise ValueError("At least one of use_aps or use_dvs must be True.")
        self.transform = transform
        self.data_file_list = glob.glob(f"{data_folder}/*.hdf5")
        self.target_transform = target_transform
        self.window_size = window_size
        self.use_aps = use_aps
        self.use_dvs = use_dvs
        self.inv_apsdvs_max = 1 / 255
        self.apsdvs_min = 0
        
        self.total_windows, self.dataset_dict = self.create_dataset_dict()

    def find_num_windows(self, path_to_file):
        # Note the integer division here, drop_last is implied true which is OK.
        with h5py.File(path_to_file, "r") as f:
            return len(f["steering_wheel_angle"]) // self.window_size
    
    def create_dataset_dict(self):
        total_windows = 0
        overall_index = 0
        dataset_dict = {}
        for data_file in self.data_file_list:
            windows_of_current_file = self.find_num_windows(data_file)
            total_windows += windows_of_current_file

            # Show which overall index corresponds to which file and window index
            for i in range(windows_of_current_file):
                dataset_dict[overall_index] = {
                    "filename" : data_file,
                    "window_index" : i
                }
                overall_index += 1

        return total_windows, dataset_dict
    
    def read_file(self, filename, window_index):
        with h5py.File(filename, "r") as f:
            if self.use_dvs:
                try:
                    dvs_frame = np.array(f[DVS_FRAME_NAME][window_index:window_index + self.window_size])
                except KeyError:
                    raise KeyError(f"Could not find dataset {DVS_FRAME_NAME} in file {filename}.")
            if self.use_aps:
                try:
                    aps_frame = np.array(f[APS_FRAME_NAME][window_index:window_index + self.window_size])
                except KeyError:
                    raise KeyError(f"Could not find dataset {APS_FRAME_NAME} in file {filename}.")
            if self.use_aps and self.use_dvs:
                # the frames are 1 x 224 x 224, we want 224 x 224 x 2 after concat
                # 3 to have it be B x H x W x C
                combined_frames = np.concatenate((dvs_frame, aps_frame), axis=0)
            
            combined_frames = combined_frames.transpose(1, 2, 0)
            steering_angles = np.array(f["steering_wheel_angle"][window_index:window_index + self.window_size])
        # print(f"Combined frames shape is {combined_frames.shape}.")
        return combined_frames, steering_angles
    
    # used to normalise the aps and dvs frames to the range [0, 1]
    def normalise(self, data):
        # shorthand for (data - self.apsdvs_min) / (self.apsdvs_max - self.apsdvs_min) since we know the min is 0
        new_data = np.empty_like(data, dtype=DTYPE)
        new_data = data * self.inv_apsdvs_max
        return new_data

    def __len__(self):
        return self.total_windows

    def __getitem__(self, index):
        filename = self.dataset_dict[index]["filename"]
        window_index = self.dataset_dict[index]["window_index"]
        data, target = self.read_file(filename, window_index)
        data = self.normalise(data)
        
        if self.transform:
            data = self.transform(data)
        
        if self.target_transform:
            target = self.target_transform(target)
        
        return data, target
        
    def print_dataset_dict(self):
        for key, value in self.dataset_dict.items():
            print(f"Index: {key}, Filename: {value['filename']}, Window Index: {value['window_index']}")

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])

train_dataset = LazyDDD20Dataset(TRAINING_DIR, transform=transform, window_size=1, use_aps=USE_APS, use_dvs=USE_DVS)
test_dataset = LazyDDD20Dataset(TEST_DIR, transform=transform, window_size=1, use_aps=USE_APS, use_dvs=USE_DVS)

# load the data, can implement batching etc. here
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
print(f"Train loader has {len(train_loader)} batches, equalling {len(train_loader) * BATCH_SIZE} frames.\n-Test loader has {len(test_loader)} batches, equalling {len(test_loader) * BATCH_SIZE} frames.")

In [9]:
for data, target in train_loader:
    pass